In [35]:
%load_ext autoreload
%autoreload 2
# %matplotlib inline

import os
while 'notebooks' in os.getcwd():
    os.chdir("../")

import torch
from torch import nn, einsum

import quantus
import gc
import torch.nn.functional as F
import pandas as pd

from lib.helpers import plot_example_grid
from lib.attributions import GradientAscentDiff, PullbackAscentDiff, \
    quantus_pullback_ascent_diff_explain_func
from lib.setup import setup_notebook
from lib.defaults import get_default_kwargs
from lib.surrogates import LayerNorm2d, PVTAttention, soften_module_inplace_
from lib.evaluator import QuantusEvaluator, default_explainers, default_metrics

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
short_metrics_map = {
    "infidelity": "Infidelity",
    "faithfulness_correlation": "Faith.Corr",
    "faithfulness_estimate": "Faith.Est",
    "monotonicity_correlation": "Mono.Corr",
    "max_sensitivity": "Max.Sens",
    "random_logit": "Rand.Logit",
}
explainers = ["SoftPullback", "PullbackAscent", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]
# explainers = ["SoftPullback", "PullbackAscent", "PullbackAscentNoAlpha", "PullbackAscent3", "SmoothPullback", "FusionPullback", "Gradient", "GradientAscent", "SmoothGrad", "FusionGrad", "GradientShap", "IntegratedGradients", "DeepLift", "GuidedGradCam"]

In [49]:
def merge_with_priority(df_big, df_small, key_col="Faith.Corr"):
    # we merge results as we re-genenerated the faithfulness_correlation metric for different set of patches
    # the previous selection had too large patches, which inflated the correlation score
    result = df_big.loc[df_small.index].copy()
    result[key_col] = df_small[key_col]
    return result

def process_df(
    df_path,
    precision=3,
    uncertainty="std",
    confidence_level=0.95,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Use uncertainty='bootstrap_ci' to report mean [95% CI] instead of mean±std."""
    selected_columns = list(short_metrics_map.keys())
    
    df = QuantusEvaluator.load_results(df_path)
    # df = df[selected_columns].rename(index=short_metrics_map, columns=short_metrics_map)
    df = df[
        [col for col in selected_columns if col in df.columns]
    ].rename(index=short_metrics_map, columns=short_metrics_map)
    # df = df.loc[explainers]
    df = df.loc[[idx for idx in explainers if idx in df.index]]
    
    print("num_samples:", len(df.iloc[0].iloc[0]))
    
    df = QuantusEvaluator.summarize_results(
        df,
        precision=precision,
        uncertainty=uncertainty,
        confidence_level=confidence_level,
        bootstrap_resamples=bootstrap_resamples,
        bootstrap_seed=bootstrap_seed,
    )
    return df


In [50]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn")
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,16.303±31.68,0.606±0.3,0.53±0.4,0.375±0.39,0.212±0.08,-0.064±0.33
PullbackAscent,5.616±13.18,0.57±0.29,0.376±0.48,0.312±0.37,0.39±0.11,0.136±0.13
SmoothPullback,14.03±30.57,0.545±0.32,0.428±0.46,0.309±0.4,0.398±0.11,-0.048±0.32
FusionPullback,16.056±34.46,0.513±0.33,0.406±0.47,0.315±0.39,0.513±0.11,-0.035±0.29
Gradient,100.09±107.94,0.457±0.35,0.466±0.41,0.316±0.37,0.898±0.14,-0.053±0.29
GradientAscent,18.523±35.24,0.508±0.3,0.264±0.46,0.234±0.32,1.217±0.06,6.65e-04±0.07
SmoothGrad,52.044±70.8,0.513±0.34,0.508±0.41,0.385±0.36,0.781±0.11,-0.025±0.19
FusionGrad,38.651±60.23,0.506±0.34,0.521±0.39,0.363±0.34,0.955±0.13,-0.013±0.16
GradientShap,76.05±94.76,0.43±0.36,0.632±0.33,0.381±0.41,1.086±0.22,-0.042±0.26
IntegratedGradients,75.796±92.29,0.434±0.36,0.634±0.33,0.379±0.41,0.752±0.15,-0.046±0.27


In [62]:
vgg_df = process_df("results/quantus_vgg_fc_nips_20_50_vgg11_bn", uncertainty="bootstrap_ci")
vgg_df.to_csv('results/vgg_df_20_50.csv', index=True)
vgg_df

num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,"16.303 [14.445, 18.407]","0.606 [0.587, 0.624]","0.53 [0.505, 0.554]","0.375 [0.351, 0.399]","0.212 [0.208, 0.217]","-0.064 [-0.084, -0.043]"
PullbackAscent,"5.616 [4.887, 6.508]","0.57 [0.552, 0.588]","0.376 [0.346, 0.405]","0.312 [0.289, 0.335]","0.39 [0.383, 0.397]","0.136 [0.128, 0.144]"
SmoothPullback,"14.03 [12.249, 15.968]","0.545 [0.525, 0.565]","0.428 [0.4, 0.455]","0.309 [0.284, 0.334]","0.398 [0.392, 0.405]","-0.048 [-0.068, -0.029]"
FusionPullback,"16.056 [14.045, 18.308]","0.513 [0.493, 0.534]","0.406 [0.377, 0.435]","0.315 [0.29, 0.339]","0.513 [0.506, 0.52]","-0.035 [-0.053, -0.018]"
Gradient,"100.09 [93.529, 107.002]","0.457 [0.435, 0.478]","0.466 [0.441, 0.491]","0.316 [0.293, 0.339]","0.898 [0.89, 0.907]","-0.053 [-0.071, -0.035]"
GradientAscent,"18.523 [16.421, 20.805]","0.508 [0.49, 0.527]","0.264 [0.236, 0.294]","0.234 [0.215, 0.254]","1.217 [1.214, 1.221]","6.65e-04 [-0.004, 0.005]"
SmoothGrad,"52.044 [47.732, 56.511]","0.513 [0.492, 0.534]","0.508 [0.482, 0.534]","0.385 [0.362, 0.407]","0.781 [0.775, 0.789]","-0.025 [-0.036, -0.012]"
FusionGrad,"38.651 [35.077, 42.557]","0.506 [0.485, 0.527]","0.521 [0.497, 0.545]","0.363 [0.342, 0.383]","0.955 [0.947, 0.963]","-0.013 [-0.023, -0.003]"
GradientShap,"76.05 [70.365, 82.049]","0.43 [0.407, 0.452]","0.632 [0.611, 0.653]","0.381 [0.356, 0.406]","1.086 [1.073, 1.099]","-0.042 [-0.058, -0.025]"
IntegratedGradients,"75.796 [70.082, 81.64]","0.434 [0.412, 0.456]","0.634 [0.613, 0.654]","0.379 [0.353, 0.403]","0.752 [0.743, 0.762]","-0.046 [-0.062, -0.03]"


In [51]:
pvt_df_fc = process_df("results/quantus_pvt_fc_nips_20_50_pvt_v2_b1")
pvt_df = process_df("results/quantus_pvt_it_nips_20_50_pvt_v2_b1")
pvt_merged_df = merge_with_priority(pvt_df, pvt_df_fc)
pvt_merged_df.to_csv('results/pvt_df_20_50.csv', index=True)
pvt_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,6.264±6.4,0.119±0.34,0.16±0.41,0.164±0.43,1.066±0.18,-0.006±0.35
PullbackAscent,1.634±1.03,0.122±0.36,0.201±0.41,0.219±0.33,0.855±0.12,0.121±0.11
SmoothPullback,4.974±5.46,0.102±0.39,0.146±0.49,0.231±0.38,0.519±0.09,0.008±0.23
FusionPullback,5.156±5.72,0.093±0.39,0.114±0.49,0.2±0.38,0.753±0.11,0.007±0.19
Gradient,8.914±7.89,0.105±0.33,0.117±0.41,0.185±0.43,1.034±0.16,-0.03±0.3
GradientAscent,4.506±4.07,0.118±0.33,0.06±0.42,0.164±0.35,1.242±0.07,0.006±0.06
SmoothGrad,8.798±7.08,0.096±0.38,0.171±0.48,0.28±0.38,0.569±0.1,-0.012±0.13
FusionGrad,6.673±5.6,0.101±0.39,0.144±0.48,0.262±0.38,0.979±0.16,5.73e-04±0.11
GradientShap,12.433±8.45,0.067±0.32,0.177±0.39,0.169±0.45,2.471±1.56,0.003±0.34
IntegratedGradients,12.578±8.39,0.068±0.32,0.157±0.4,0.17±0.45,1.374±0.37,0.008±0.33


In [52]:
resnet_df_fc = process_df("results/quantus_resnet_fc_nips_20_50_resnet50")
resnet_df = process_df("results/quantus_resnet_it_nips_20_50_resnet50")
resnet_merged_df = merge_with_priority(resnet_df, resnet_df_fc)
resnet_merged_df.to_csv('results/resnet_df_20_50.csv', index=True)
resnet_merged_df

num_samples: 1000
num_samples: 1000


,Infidelity,Faith.Corr,Faith.Est,Mono.Corr,Max.Sens,Rand.Logit
SoftPullback,5.989±5.67,0.389±0.37,0.437±0.41,0.42±0.39,0.119±0.04,-0.062±0.42
PullbackAscent,5.384±4.83,0.382±0.37,0.394±0.43,0.421±0.4,0.244±0.09,0.212±0.19
SmoothPullback,8.122±10.13,0.326±0.39,0.321±0.46,0.352±0.4,0.22±0.05,-0.065±0.4
FusionPullback,10.131±14.55,0.306±0.4,0.306±0.47,0.35±0.4,0.362±0.08,-0.051±0.39
Gradient,83.294±68.15,0.241±0.36,0.289±0.45,0.275±0.39,0.952±0.17,-0.032±0.23
GradientAscent,29.48±35.36,0.278±0.37,0.131±0.46,0.231±0.32,1.267±0.04,0.002±0.05
SmoothGrad,66.883±58.16,0.331±0.4,0.321±0.49,0.364±0.36,0.639±0.08,-0.024±0.2
FusionGrad,58.514±57.8,0.344±0.39,0.336±0.48,0.357±0.35,0.855±0.09,-0.014±0.17
GradientShap,69.072±62.93,0.233±0.38,0.409±0.44,0.344±0.41,1.4±0.41,-0.023±0.24
IntegratedGradients,66.83±61.21,0.244±0.38,0.421±0.44,0.35±0.41,0.788±0.21,-0.029±0.23


In [53]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# NumPy 2 pickles refer to numpy._core. Expose equivalent modules when this
# notebook runs with NumPy 1 so that saved results remain readable.
if int(np.__version__.split(".", maxsplit=1)[0]) < 2:
    import numpy.core as _numpy_core
    import numpy.core.multiarray as _numpy_multiarray
    import numpy.core.numeric as _numpy_numeric

    sys.modules.setdefault("numpy._core", _numpy_core)
    sys.modules.setdefault("numpy._core.multiarray", _numpy_multiarray)
    sys.modules.setdefault("numpy._core.numeric", _numpy_numeric)

def _as_float_array(value):
    if isinstance(value, (list, tuple, np.ndarray, pd.Series)):
        arr = np.asarray(value, dtype=float).reshape(-1)
    else:
        arr = np.asarray([value], dtype=float)
    return arr[np.isfinite(arr)]

def _format_val(x, precision=3):
    if np.isnan(x):
        return ""
    if abs(x) >= 1e4 or (abs(x) > 0 and abs(x) < 1e-3):
        return f"{x:.2e}"
    s = f"{x:.{precision}f}"
    return s.rstrip("0").rstrip(".") if "." in s else s

def _format_mean_std(arr, precision=3):
    if arr.size == 0:
        return ""
    mean = arr.mean()
    std = arr.std()
    mean_str = _format_val(mean, precision=precision)
    std_str = _format_val(std, precision=max(1, precision - 1))
    return f"{mean_str}±{std_str}"

def summarize_ablations_pickle(pkl_path, show=True):
    pkl_path = Path(pkl_path)
    if not pkl_path.exists():
        raise FileNotFoundError(f"File not found: {pkl_path}")

    df = pd.read_pickle(pkl_path)
    meta_cols = ["model_name", "parameter", "value", "default_value", "explainer"]
    metric_cols = [
        col
        for col in df.columns
        if col not in meta_cols and not col.startswith("is_default")
    ]
    group_cols = ["model_name", "parameter", "value", "default_value", "explainer"]

    summary_rows = []
    for _, row in df.iterrows():
        base = {col: row[col] for col in group_cols}
        for metric in metric_cols:
            arr = _as_float_array(row[metric])
            base[metric] = _format_mean_std(arr, precision=3)
        summary_rows.append(base)

    summary = (
        pd.DataFrame(summary_rows)
        .sort_values(["model_name", "parameter", "value", "explainer"])
        .reset_index(drop=True)
    )

    csv_path = pkl_path.with_name(pkl_path.stem + "_summary.csv")
    summary.to_csv(csv_path, index=False)

    print(f"Loaded: {pkl_path}")
    print(f"Saved CSV: {csv_path}")
    if show:
        display(summary)
    return summary, csv_path

def summarize_focus_pickle(focus_pkl_path, show=True):
    focus_pkl_path = Path(focus_pkl_path)
    if not focus_pkl_path.exists():
        results_dir = focus_pkl_path.parent
        available = [p.name for p in sorted(results_dir.glob("focus_model_name=*.pkl"))]
        raise FileNotFoundError(
            f"File not found: {focus_pkl_path}. Available Focus files: {available}"
        )

    focus_df = pd.read_pickle(focus_pkl_path)
    required_cols = {"explainer", "score"}
    missing_cols = required_cols.difference(focus_df.columns)
    if missing_cols:
        raise ValueError(
            f"Missing required columns in Focus results: {sorted(missing_cols)}"
        )

    focus_summary = (
        focus_df.groupby("explainer", dropna=False)["score"]
        .agg(["mean", "std", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )

    focus_csv_path = focus_pkl_path.with_name(focus_pkl_path.stem + "_summary.csv")
    focus_summary.to_csv(focus_csv_path, index=False)

    print(f"Loaded: {focus_pkl_path}")
    print(f"Saved CSV: {focus_csv_path}")
    if show:
        display(focus_summary)
    return focus_summary, focus_csv_path

def _bootstrap_mean_ci(
    values,
    rng,
    n_resamples=10_000,
    confidence_level=0.95,
    chunk_size=1_000,
):
    """Return a percentile bootstrap CI for the mean of paired differences."""
    values = np.asarray(values, dtype=float).reshape(-1)
    if values.size == 0:
        return np.nan, np.nan
    if not 0 < confidence_level < 1:
        raise ValueError("confidence_level must be between 0 and 1.")
    if n_resamples < 1:
        raise ValueError("n_resamples must be at least 1.")

    bootstrap_means = np.empty(n_resamples, dtype=float)
    for start in range(0, n_resamples, chunk_size):
        stop = min(start + chunk_size, n_resamples)
        indices = rng.integers(0, values.size, size=(stop - start, values.size))
        bootstrap_means[start:stop] = values[indices].mean(axis=1)

    alpha = (1 - confidence_level) / 2
    return tuple(np.quantile(bootstrap_means, [alpha, 1 - alpha]))

def paired_differences_vs_default(
    ablations_pkl_path,
    show=True,
    explainers=("PullbackAscent", "SoftPullback"),
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    ablations_pkl_path = Path(ablations_pkl_path)
    if not ablations_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {ablations_pkl_path}")

    df = pd.read_pickle(ablations_pkl_path)
    df = df[df["explainer"].isin(explainers)].copy()
    meta_cols = {"model_name", "parameter", "value", "default_value", "explainer", "is_default"}
    metric_cols = [col for col in df.columns if col not in meta_cols]
    rng = np.random.default_rng(bootstrap_seed)

    rows = []
    default_summary_rows = []
    for (model_name, explainer), df_expl in df.groupby(
        ["model_name", "explainer"], sort=False
    ):
        if "is_default" in df_expl.columns and df_expl["is_default"].any():
            default_rows = df_expl[df_expl["is_default"] == True]
        else:
            default_rows = df_expl[
                (df_expl["parameter"] == "default") & (df_expl["value"] == "default")
            ]

        if len(default_rows) != 1:
            raise ValueError(
                f"Expected exactly one default row for model={model_name}, "
                f"explainer={explainer}, got {len(default_rows)}"
            )

        default_row = default_rows.iloc[0]
        for metric in metric_cols:
            default_values = _as_float_array(default_row[metric])
            default_summary_rows.append({
                "model_name": model_name,
                "explainer": explainer,
                "metric": metric,
                "default_mean": float(default_values.mean()),
                "default_std": (
                    float(default_values.std(ddof=1))
                    if default_values.size > 1 else np.nan
                ),
                "count": int(default_values.size),
            })

        for _, row in df_expl.iterrows():
            is_default_row = bool(row.get("is_default", False)) or (
                row["parameter"] == "default" and row["value"] == "default"
            )
            if is_default_row:
                continue

            setting = f"{row['parameter']}={row['value']}"
            for metric in metric_cols:
                x = np.asarray(row[metric], dtype=float).reshape(-1)
                y = np.asarray(default_row[metric], dtype=float).reshape(-1)
                if x.shape != y.shape:
                    raise ValueError(
                        f"Cannot pair metric={metric}, explainer={explainer}, "
                        f"setting={setting}: variant has {x.size} values and "
                        f"default has {y.size}."
                    )
                valid = np.isfinite(x) & np.isfinite(y)
                x = x[valid]
                y = y[valid]
                n = len(x)
                if n == 0:
                    continue

                diff = x - y
                ci95_low, ci95_high = _bootstrap_mean_ci(
                    diff,
                    rng=rng,
                    n_resamples=bootstrap_resamples,
                )
                rows.append(
                    {
                        "model_name": row["model_name"],
                        "explainer": explainer,
                        "setting": setting,
                        "parameter": row["parameter"],
                        "value": row["value"],
                        "default_value": row["default_value"],
                        "metric": metric,
                        "default_mean": float(np.mean(y)),
                        "variant_mean": float(np.mean(x)),
                        "delta_mean": float(np.mean(diff)),
                        "delta_std": float(np.std(diff, ddof=1)) if n > 1 else np.nan,
                        "ci95_low": float(ci95_low),
                        "ci95_high": float(ci95_high),
                        "paired_count": int(n),
                    }
                )

    paired_df = pd.DataFrame(rows).sort_values(
        ["explainer", "parameter", "value", "metric"]
    ).reset_index(drop=True)

    paired_csv_path = ablations_pkl_path.with_name(
        ablations_pkl_path.stem + "_paired_diff_vs_default.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    metric_directions = {
        "infidelity": "↓",
        "faithfulness_correlation": "↑",
        "faithfulness_estimate": "↑",
        "monotonicity_correlation": "↑",
        "max_sensitivity": "↓",
        "random_logit": "↓",
    }
    metric_labels = {
        metric: f"{short_metrics_map.get(metric, metric)} {metric_directions.get(metric, '')}".rstrip()
        for metric in metric_cols
    }

    paired_display = paired_df.copy()
    paired_display["metric"] = paired_display["metric"].map(metric_labels)
    paired_display["delta_ci95"] = paired_display.apply(
        lambda r: (
            f"{r['delta_mean']:.4f} "
            f"[{r['ci95_low']:.4f}, {r['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_pivot = (
        paired_display.assign(
            value_num=pd.to_numeric(paired_display["value"], errors="coerce")
        )
        .sort_values(["explainer", "parameter", "value_num", "value"])
        .pivot_table(
            index=["explainer", "parameter", "default_value", "value"],
            columns="metric",
            values="delta_ci95",
            aggfunc="first",
        )
        .reset_index()
    )

    default_summary = pd.DataFrame(default_summary_rows)
    default_summary["metric"] = default_summary["metric"].map(metric_labels)
    default_summary["mean_std"] = default_summary.apply(
        lambda r: f"{r['default_mean']:.4f}±{r['default_std']:.4f}",
        axis=1,
    )
    default_pivot = default_summary.pivot(
        index=["model_name", "explainer"],
        columns="metric",
        values="mean_std",
    ).reset_index()
    default_pivot.columns.name = None

    print(f"Loaded: {ablations_pkl_path}")
    print(f"Saved paired difference CSV: {paired_csv_path}")
    print("Default scores (mean±std):")
    if show:
        display(default_pivot)
    print(
        f"Paired delta format: variant - default, mean [95% percentile bootstrap CI] "
        f"using {bootstrap_resamples:,} resamples (seed={bootstrap_seed})."
    )
    print(f"Paired counts present in the detailed CSV: {sorted(paired_df['paired_count'].unique())}")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

EXPLAINER_PAIRS = (
    ("SoftPullback", "Gradient", "SoftPullback - Gradient"),
    ("PullbackAscent", "GradientAscent", "PullbackAscent - GradientAscent"),
    ("SmoothPullback", "SmoothGrad", "SmoothPullback - SmoothGrad"),
    ("FusionPullback", "FusionGrad", "FusionPullback - FusionGrad"),
)

def paired_explainer_differences(
    results_pkl_path,
    metric_overrides=None,
    pairs=EXPLAINER_PAIRS,
    show=True,
    bootstrap_resamples=10_000,
    bootstrap_seed=314,
):
    """Compute per-image Pullback-minus-Gradient contrasts within each run."""
    results_pkl_path = Path(results_pkl_path)
    if not results_pkl_path.exists():
        raise FileNotFoundError(f"File not found: {results_pkl_path}")

    base_df = pd.read_pickle(results_pkl_path)
    metric_frames = {metric: base_df for metric in base_df.columns}
    metric_sources = {metric: results_pkl_path for metric in base_df.columns}

    for metric, override_path in (metric_overrides or {}).items():
        override_path = Path(override_path)
        if not override_path.exists():
            raise FileNotFoundError(f"File not found: {override_path}")
        override_df = pd.read_pickle(override_path)
        if metric not in override_df.columns:
            raise ValueError(f"Metric {metric!r} is missing from {override_path}.")
        metric_frames[metric] = override_df
        metric_sources[metric] = override_path

    rng = np.random.default_rng(bootstrap_seed)
    rows = []
    for pullback_name, gradient_name, pair_label in pairs:
        for metric, metric_df in metric_frames.items():
            missing = {pullback_name, gradient_name}.difference(metric_df.index)
            if missing:
                raise ValueError(
                    f"Missing explainers {sorted(missing)} for metric={metric!r} "
                    f"in {metric_sources[metric]}."
                )

            pullback_values = np.asarray(
                metric_df.at[pullback_name, metric], dtype=float
            ).reshape(-1)
            gradient_values = np.asarray(
                metric_df.at[gradient_name, metric], dtype=float
            ).reshape(-1)
            if pullback_values.shape != gradient_values.shape:
                raise ValueError(
                    f"Cannot pair {pair_label}, metric={metric!r}: Pullback has "
                    f"{pullback_values.size} values and Gradient has "
                    f"{gradient_values.size}."
                )

            valid = np.isfinite(pullback_values) & np.isfinite(gradient_values)
            paired_pullback = pullback_values[valid]
            paired_gradient = gradient_values[valid]
            differences = paired_pullback - paired_gradient
            if differences.size == 0:
                continue

            ci95_low, ci95_high = _bootstrap_mean_ci(
                differences,
                rng=rng,
                n_resamples=bootstrap_resamples,
            )
            rows.append({
                "result_set": results_pkl_path.stem,
                "pair": pair_label,
                "pullback_explainer": pullback_name,
                "gradient_explainer": gradient_name,
                "metric": metric,
                "metric_source": metric_sources[metric].name,
                "pullback_mean": float(paired_pullback.mean()),
                "gradient_mean": float(paired_gradient.mean()),
                "delta_mean": float(differences.mean()),
                "delta_std": (
                    float(differences.std(ddof=1))
                    if differences.size > 1 else np.nan
                ),
                "ci95_low": float(ci95_low),
                "ci95_high": float(ci95_high),
                "paired_count": int(differences.size),
            })

    paired_df = pd.DataFrame(rows).sort_values(["pair", "metric"]).reset_index(drop=True)
    paired_csv_path = results_pkl_path.with_name(
        results_pkl_path.stem + "_paired_explainers.csv"
    )
    paired_df.to_csv(paired_csv_path, index=False)

    metric_directions = {
        "infidelity": "↓",
        "faithfulness_correlation": "↑",
        "faithfulness_estimate": "↑",
        "monotonicity_correlation": "↑",
        "max_sensitivity": "↓",
        "random_logit": "↓",
    }
    metric_labels = {
        metric: f"{short_metrics_map.get(metric, metric)} {metric_directions.get(metric, '')}".rstrip()
        for metric in paired_df["metric"].unique()
    }
    paired_display = paired_df.copy()
    paired_display["metric"] = paired_display["metric"].map(metric_labels)
    paired_display["delta_ci95"] = paired_display.apply(
        lambda row: (
            f"{row['delta_mean']:.4f} "
            f"[{row['ci95_low']:.4f}, {row['ci95_high']:.4f}]"
        ),
        axis=1,
    )
    paired_pivot = paired_display.pivot(
        index="pair",
        columns="metric",
        values="delta_ci95",
    ).reset_index()
    paired_pivot.columns.name = None

    print(f"Loaded base results: {results_pkl_path}")
    for metric, source_path in metric_sources.items():
        print(f"Metric source [{metric}]: {source_path}")
    print(f"Saved paired explainer CSV: {paired_csv_path}")
    print(
        f"Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] "
        f"using {bootstrap_resamples:,} resamples (seed={bootstrap_seed})."
    )
    print(f"Paired counts: {sorted(paired_df['paired_count'].unique())}")
    if show:
        display(paired_pivot)
    return paired_df, paired_pivot, paired_csv_path

In [54]:
summary_resnet_ablations, csv_resnet_ablations = summarize_ablations_pickle(
    "results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)

Loaded: results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl
Saved CSV: results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,random_logit
0,resnet50,K,1,5,PullbackAscent,6.024±5.72,0.398±0.39,0.466±0.39,-0.019±0.43
1,resnet50,K,2,5,PullbackAscent,5.534±5.1,0.405±0.39,0.433±0.41,0.152±0.31
2,resnet50,K,3,5,PullbackAscent,5.37±4.91,0.398±0.39,0.419±0.42,0.199±0.26
3,resnet50,K,10,5,PullbackAscent,5.652±5.79,0.354±0.38,0.358±0.46,0.214±0.14
4,resnet50,alpha,5,20,PullbackAscent,5.459±5.02,0.409±0.38,0.415±0.41,0.266±0.24
5,resnet50,alpha,10,20,PullbackAscent,5.357±4.89,0.397±0.38,0.404±0.42,0.263±0.21
6,resnet50,alpha,40,20,PullbackAscent,5.68±7.27,0.362±0.38,0.377±0.46,0.17±0.18
7,resnet50,default,default,default,PullbackAscent,5.338±4.86,0.381±0.38,0.389±0.45,0.228±0.2
8,resnet50,default,default,default,SoftPullback,6.024±5.75,0.398±0.39,0.466±0.39,-0.019±0.43
9,resnet50,tau_maxpool,0.01,0.3,PullbackAscent,5.339±4.86,0.364±0.39,0.328±0.48,0.196±0.18


In [55]:
summary_resnet_default, csv_resnet_default = summarize_ablations_pickle(
    "results/quantus_ablations_default_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)

Loaded: results/quantus_ablations_default_model_name=resnet50_n_batches=25_test_mode=test.pkl
Saved CSV: results/quantus_ablations_default_model_name=resnet50_n_batches=25_test_mode=test_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,random_logit
0,resnet50,default,default,default,PullbackAscent,5.338±4.86,0.381±0.38,0.39±0.45,0.228±0.2
1,resnet50,default,default,default,SoftPullback,6.024±5.75,0.398±0.39,0.466±0.39,-0.019±0.43


In [56]:
summary_pvt_ablations, csv_pvt_ablations = summarize_ablations_pickle(
    "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
)

Loaded: results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl
Saved CSV: results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test_summary.csv


,model_name,parameter,value,default_value,explainer,infidelity,faithfulness_correlation,faithfulness_estimate,random_logit
0,pvt_v2_b1,K,1,5,PullbackAscent,6.44±6.2,0.097±0.33,0.152±0.39,-0.018±0.38
1,pvt_v2_b1,K,2,5,PullbackAscent,2.8±3.01,0.124±0.36,0.198±0.41,0.04±0.21
2,pvt_v2_b1,K,3,5,PullbackAscent,1.883±1.26,0.134±0.37,0.209±0.42,0.07±0.16
3,pvt_v2_b1,K,10,5,PullbackAscent,1.571±1.14,0.12±0.36,0.184±0.42,0.139±0.07
4,pvt_v2_b1,alpha,5,20,PullbackAscent,1.993±1.68,0.134±0.36,0.144±0.43,0.107±0.13
5,pvt_v2_b1,alpha,10,20,PullbackAscent,1.737±1.21,0.133±0.37,0.176±0.43,0.113±0.12
6,pvt_v2_b1,alpha,40,20,PullbackAscent,1.657±1.32,0.136±0.37,0.234±0.41,0.104±0.11
7,pvt_v2_b1,default,default,default,PullbackAscent,1.693±1.41,0.133±0.37,0.202±0.43,0.111±0.11
8,pvt_v2_b1,default,default,default,SoftPullback,6.348±5.98,0.097±0.33,0.152±0.39,-0.018±0.38
9,pvt_v2_b1,tau_attention,0.5,1.0,PullbackAscent,1.743±1.3,0.13±0.36,0.195±0.43,0.095±0.1


In [57]:
focus_summary_resnet, focus_csv_resnet = summarize_focus_pickle(
    "results/focus_model_name=resnet50.pkl"
)

Loaded: results/focus_model_name=resnet50.pkl
Saved CSV: results/focus_model_name=resnet50_summary.csv


,explainer,mean,std,count
0,GuidedGradCam,0.849295,0.176164,500
1,SoftPullback,0.748278,0.134405,500
2,PullbackAscent,0.730507,0.129333,500
3,SmoothPullback,0.718342,0.149743,500
4,FusionPullback,0.696509,0.150471,500
5,SmoothGrad,0.640846,0.123955,500
6,FusionGrad,0.634871,0.119925,500
7,Gradient,0.633657,0.139535,500
8,IntegratedGradients,0.622721,0.167035,500
9,DeepLift,0.621046,0.175279,500


In [58]:
focus_summary_pvt, focus_csv_pvt = summarize_focus_pickle(
    "results/focus_model_name=pvt_v2_b1.pkl"
)

Loaded: results/focus_model_name=pvt_v2_b1.pkl
Saved CSV: results/focus_model_name=pvt_v2_b1_summary.csv


,explainer,mean,std,count
0,SmoothPullback,0.634890,0.102580,500
1,SoftPullback,0.624963,0.164957,500
2,FusionPullback,0.615664,0.104791,500
3,FusionGrad,0.605290,0.104144,500
4,SmoothGrad,0.592187,0.111431,500
5,Gradient,0.560886,0.170342,500
6,IntegratedGradients,0.559527,0.177091,500
7,GradientShap,0.553672,0.179238,500
8,PullbackAscent,0.553121,0.094839,500
9,DeepLift,0.550655,0.203432,500


In [47]:
paired_df_resnet, paired_pivot_resnet, paired_csv_resnet = paired_differences_vs_default(
    "results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl"
)
# paired_df_pvt, paired_pivot_pvt, paired_csv_pvt = paired_differences_vs_default(
#     "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
# )

Loaded: results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test.pkl
Saved paired difference CSV: results/quantus_ablations_model_name=resnet50_n_batches=25_test_mode=test_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,resnet50,PullbackAscent,0.3814±0.3851,0.3895±0.4508,5.3383±4.8617,0.2277±0.1960
1,resnet50,SoftPullback,0.3979±0.3910,0.4657±0.3944,6.0244±5.7544,-0.0187±0.4262


Paired delta format: variant - default, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts present in the detailed CSV: [500]


metric,explainer,parameter,default_value,value,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,PullbackAscent,K,5.0,1.00,"0.0165 [0.0052, 0.0274]","0.0761 [0.0564, 0.0961]","0.6854 [0.4286, 0.9590]","-0.2464 [-0.2690, -0.2238]"
1,PullbackAscent,K,5.0,2.00,"0.0234 [0.0161, 0.0308]","0.0431 [0.0281, 0.0582]","0.1958 [0.0682, 0.3267]","-0.0758 [-0.0873, -0.0641]"
2,PullbackAscent,K,5.0,3.00,"0.0165 [0.0119, 0.0210]","0.0300 [0.0184, 0.0417]","0.0318 [-0.0389, 0.1005]","-0.0284 [-0.0346, -0.0223]"
3,PullbackAscent,K,5.0,10.00,"-0.0270 [-0.0338, -0.0201]","-0.0318 [-0.0461, -0.0175]","0.3136 [0.1207, 0.6217]","-0.0133 [-0.0196, -0.0072]"
4,PullbackAscent,alpha,20.0,5.00,"0.0277 [0.0204, 0.0351]","0.0260 [0.0113, 0.0409]","0.1204 [-0.0011, 0.2424]","0.0380 [0.0303, 0.0457]"
5,PullbackAscent,alpha,20.0,10.00,"0.0158 [0.0113, 0.0202]","0.0150 [0.0039, 0.0261]","0.0189 [-0.0491, 0.0851]","0.0354 [0.0314, 0.0394]"
6,PullbackAscent,alpha,20.0,40.00,"-0.0190 [-0.0241, -0.0138]","-0.0125 [-0.0255, 0.0002]","0.3421 [0.0497, 0.8497]","-0.0581 [-0.0616, -0.0544]"
7,PullbackAscent,tau_maxpool,0.3,0.01,"-0.0177 [-0.0235, -0.0119]","-0.0618 [-0.0802, -0.0434]","0.0011 [-0.0572, 0.0575]","-0.0313 [-0.0350, -0.0277]"
8,PullbackAscent,tau_maxpool,0.3,0.50,"0.0043 [0.0026, 0.0061]","0.0194 [0.0108, 0.0281]","0.0580 [0.0346, 0.0860]","0.0027 [0.0014, 0.0040]"
9,PullbackAscent,tau_relu,0.6,0.30,"-0.0226 [-0.0375, -0.0077]","-0.0484 [-0.0753, -0.0213]","3.0005 [2.3373, 3.7326]","-0.1563 [-0.1688, -0.1440]"


In [48]:
paired_df_pvt, paired_pivot_pvt, paired_csv_pvt = paired_differences_vs_default(
    "results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl"
)

Loaded: results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test.pkl
Saved paired difference CSV: results/quantus_ablations_model_name=pvt_v2_b1_n_batches=25_test_mode=test_paired_diff_vs_default.csv
Default scores (mean±std):


,model_name,explainer,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,pvt_v2_b1,PullbackAscent,0.1328±0.3714,0.2015±0.4263,1.6928±1.4162,0.1110±0.1119
1,pvt_v2_b1,SoftPullback,0.0966±0.3304,0.1518±0.3912,6.3481±5.9812,-0.0178±0.3799


Paired delta format: variant - default, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts present in the detailed CSV: [500]


metric,explainer,parameter,default_value,value,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Rand.Logit ↓
0,PullbackAscent,K,5.0,1.0,"-0.0362 [-0.0651, -0.0066]","-0.0498 [-0.0868, -0.0116]","4.7475 [4.2074, 5.3071]","-0.1288 [-0.1550, -0.1029]"
1,PullbackAscent,K,5.0,2.0,"-0.0085 [-0.0262, 0.0097]","-0.0033 [-0.0296, 0.0232]","1.1071 [0.8526, 1.3782]","-0.0708 [-0.0813, -0.0603]"
2,PullbackAscent,K,5.0,3.0,"0.0010 [-0.0101, 0.0119]","0.0074 [-0.0112, 0.0266]","0.1901 [0.1062, 0.2648]","-0.0405 [-0.0457, -0.0353]"
3,PullbackAscent,K,5.0,10.0,"-0.0126 [-0.0246, -0.0005]","-0.0179 [-0.0389, 0.0033]","-0.1222 [-0.2156, -0.0520]","0.0284 [0.0242, 0.0326]"
4,PullbackAscent,alpha,20.0,5.0,"0.0011 [-0.0121, 0.0148]","-0.0578 [-0.0831, -0.0329]","0.3007 [0.1842, 0.4328]","-0.0035 [-0.0071, 0.0001]"
5,PullbackAscent,alpha,20.0,10.0,"0.0003 [-0.0071, 0.0079]","-0.0251 [-0.0425, -0.0079]","0.0442 [-0.0179, 0.0943]","0.0019 [0.0000, 0.0037]"
6,PullbackAscent,alpha,20.0,40.0,"0.0031 [-0.0067, 0.0127]","0.0326 [0.0141, 0.0515]","-0.0355 [-0.1040, 0.0122]","-0.0068 [-0.0087, -0.0049]"
7,PullbackAscent,tau_attention,1.0,0.5,"-0.0028 [-0.0125, 0.0070]","-0.0065 [-0.0251, 0.0124]","0.0500 [0.0115, 0.0854]","-0.0159 [-0.0206, -0.0114]"
8,PullbackAscent,tau_attention,1.0,5.0,"-0.0149 [-0.0312, 0.0010]","-0.0172 [-0.0391, 0.0054]","-0.0764 [-0.1345, -0.0285]","-0.0014 [-0.0084, 0.0054]"
9,PullbackAscent,tau_gelu,1.0,0.5,"0.0008 [-0.0074, 0.0089]","-0.0197 [-0.0390, -0.0004]","-0.0647 [-0.1492, -0.0032]","-0.0027 [-0.0070, 0.0015]"


## Paired Pullback–Gradient explainer comparisons

Each contrast is computed within one `QuantusEvaluator` result file. `evaluate_loader` evaluates every explainer on the same batch before advancing the loader, so positions in the score arrays refer to the same images. For ResNet and PVT, the corrected Faithfulness Correlation is taken from the corresponding `*_fc_*` run, and both members of a pair are always read from that same run. Entries report `Pullback - Gradient` as a mean paired difference with a 95% percentile bootstrap confidence interval.

In [59]:
paired_explainers_vgg, paired_explainers_vgg_table, paired_explainers_vgg_csv = (
    paired_explainer_differences(
        "results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl"
    )
)

Loaded base results: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [infidelity]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [faithfulness_correlation]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [faithfulness_estimate]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [monotonicity_correlation]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [max_sensitivity]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Metric source [random_logit]: results/quantus_vgg_fc_nips_20_50_vgg11_bn.pkl
Saved paired explainer CSV: results/quantus_vgg_fc_nips_20_50_vgg11_bn_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [1000]


,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,FusionPullback - FusionGrad,"0.0070 [-0.0031, 0.0170]","-0.1152 [-0.1330, -0.0974]","-22.5950 [-26.1671, -19.2468]","-0.4421 [-0.4479, -0.4363]","-0.0483 [-0.0627, -0.0338]","-0.0224 [-0.0346, -0.0101]"
1,PullbackAscent - GradientAscent,"0.0618 [0.0505, 0.0734]","0.1115 [0.0842, 0.1392]","-12.9064 [-15.1549, -10.7841]","-0.8275 [-0.8320, -0.8230]","0.0778 [0.0566, 0.0995]","0.1354 [0.1304, 0.1404]"
2,SmoothPullback - SmoothGrad,"0.0322 [0.0216, 0.0431]","-0.0801 [-0.0977, -0.0625]","-38.0143 [-42.3591, -33.8440]","-0.3831 [-0.3867, -0.3794]","-0.0752 [-0.0906, -0.0599]","-0.0237 [-0.0376, -0.0099]"
3,SoftPullback - Gradient,"0.1489 [0.1319, 0.1657]","0.0641 [0.0377, 0.0907]","-83.7863 [-90.4122, -77.2206]","-0.6860 [-0.6922, -0.6799]","0.0589 [0.0364, 0.0815]","-0.0101 [-0.0207, 0.0006]"


In [60]:
paired_explainers_resnet, paired_explainers_resnet_table, paired_explainers_resnet_csv = (
    paired_explainer_differences(
        "results/quantus_resnet_it_nips_20_50_resnet50.pkl",
        metric_overrides={
            "faithfulness_correlation": (
                "results/quantus_resnet_fc_nips_20_50_resnet50.pkl"
            ),
        },
    )
)

Loaded base results: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [infidelity]: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [faithfulness_correlation]: results/quantus_resnet_fc_nips_20_50_resnet50.pkl
Metric source [faithfulness_estimate]: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [monotonicity_correlation]: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [max_sensitivity]: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Metric source [random_logit]: results/quantus_resnet_it_nips_20_50_resnet50.pkl
Saved paired explainer CSV: results/quantus_resnet_it_nips_20_50_resnet50_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [1000]


,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,FusionPullback - FusionGrad,"-0.0385 [-0.0537, -0.0233]","-0.0294 [-0.0532, -0.0056]","-48.3830 [-51.9253, -44.9511]","-0.4932 [-0.4997, -0.4867]","-0.0073 [-0.0252, 0.0100]","-0.0372 [-0.0587, -0.0158]"
1,PullbackAscent - GradientAscent,"0.1040 [0.0874, 0.1208]","0.2631 [0.2339, 0.2916]","-24.0957 [-26.3416, -21.9837]","-1.0230 [-1.0278, -1.0182]","0.1903 [0.1655, 0.2149]","0.2105 [0.1999, 0.2210]"
2,SmoothPullback - SmoothGrad,"-0.0058 [-0.0217, 0.0094]","-0.0008 [-0.0259, 0.0239]","-58.7615 [-62.4075, -55.3068]","-0.4187 [-0.4232, -0.4142]","-0.0128 [-0.0314, 0.0053]","-0.0405 [-0.0623, -0.0187]"
3,SoftPullback - Gradient,"0.1472 [0.1226, 0.1710]","0.1475 [0.1170, 0.1783]","-77.3050 [-81.3633, -73.2595]","-0.8335 [-0.8434, -0.8237]","0.1444 [0.1164, 0.1721]","-0.0291 [-0.0509, -0.0067]"


In [63]:
paired_explainers_pvt, paired_explainers_pvt_table, paired_explainers_pvt_csv = (
    paired_explainer_differences(
        "results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl",
        metric_overrides={
            "faithfulness_correlation": (
                "results/quantus_pvt_fc_nips_20_50_pvt_v2_b1.pkl"
            ),
        },
    )
)

Loaded base results: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [infidelity]: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [faithfulness_correlation]: results/quantus_pvt_fc_nips_20_50_pvt_v2_b1.pkl
Metric source [faithfulness_estimate]: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [monotonicity_correlation]: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [max_sensitivity]: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Metric source [random_logit]: results/quantus_pvt_it_nips_20_50_pvt_v2_b1.pkl
Saved paired explainer CSV: results/quantus_pvt_it_nips_20_50_pvt_v2_b1_paired_explainers.csv
Delta format: Pullback - Gradient, mean [95% percentile bootstrap CI] using 10,000 resamples (seed=314).
Paired counts: [1000]


,pair,Faith.Corr ↑,Faith.Est ↑,Infidelity ↓,Max.Sens ↓,Mono.Corr ↑,Rand.Logit ↓
0,FusionPullback - FusionGrad,"-0.0088 [-0.0178, 0.0002]","-0.0296 [-0.0486, -0.0106]","-1.5167 [-1.9709, -1.0603]","-0.2258 [-0.2365, -0.2151]","-0.0617 [-0.0774, -0.0462]","0.0061 [-0.0052, 0.0177]"
1,PullbackAscent - GradientAscent,"0.0044 [-0.0109, 0.0199]","0.1415 [0.1196, 0.1642]","-2.8716 [-3.1287, -2.6310]","-0.3875 [-0.3924, -0.3824]","0.0544 [0.0365, 0.0730]","0.1158 [0.1088, 0.1229]"
2,SmoothPullback - SmoothGrad,"0.0059 [-0.0050, 0.0167]","-0.0255 [-0.0461, -0.0050]","-3.8244 [-4.3190, -3.3446]","-0.0497 [-0.0540, -0.0454]","-0.0483 [-0.0641, -0.0324]","0.0200 [0.0055, 0.0345]"
3,SoftPullback - Gradient,"0.0137 [0.0005, 0.0270]","0.0426 [0.0245, 0.0609]","-2.6502 [-3.1834, -2.1285]","0.0317 [0.0190, 0.0439]","-0.0208 [-0.0394, -0.0023]","0.0232 [-0.0030, 0.0496]"
